# 05. 메모리와 스트리밍

LangChain v1 에이전트의 **메모리 시스템**과 **스트리밍 모드**를 학습합니다.

## 학습 목표

- **단기 메모리(Short-term Memory):** `InMemorySaver`를 사용하여 `thread_id` 기반으로 대화 상태를 유지하는 방법을 이해합니다.
- **장기 메모리(Long-term Memory):** `InMemoryStore`를 사용하여 대화 간에 지속되는 메모리를 구현합니다.
- **메시지 트리밍:** 긴 대화에서 토큰 예산 내로 메시지를 제한하는 방법을 배웁니다.
- **스트리밍 모드:** `values`, `updates`, `messages`, `custom` 등 다양한 스트리밍 모드의 차이를 이해합니다.

## 5.1 환경 설정

In [1]:
from dotenv import load_dotenv
import os
load_dotenv(override=True)

from langchain_openai import ChatOpenAI

model = ChatOpenAI(
    model="gpt-4.1",
)

print("모델 준비 완료:", model.model_name)

모델 준비 완료: gpt-4.1


In [2]:
# Observability 설정 (선택) - LangSmith 또는 Langfuse
# .env에 키를 설정하거나, 아래 주석을 해제하여 직접 입력하세요.
# os.environ["LANGFUSE_SECRET_KEY"] = "sk-lf-..."
# os.environ["LANGFUSE_PUBLIC_KEY"] = "pk-lf-..."
# os.environ["LANGFUSE_HOST"] = "https://lf.ddok.ai"
import os

# LangSmith: LANGSMITH_TRACING=true 시 자동 활성화 (코드 수정 불필요)
if os.environ.get("LANGSMITH_TRACING", "").lower() == "true":
    os.environ.setdefault("LANGCHAIN_TRACING_V2", "true")
    os.environ.setdefault("LANGCHAIN_API_KEY", os.environ.get("LANGSMITH_API_KEY", ""))
    os.environ.setdefault("LANGCHAIN_PROJECT", os.environ.get("LANGSMITH_PROJECT", "default"))
    print(f"LangSmith tracing ON \u2014 project: {os.environ['LANGCHAIN_PROJECT']}")

# Langfuse: invoke/stream 호출 시 config={"callbacks": [langfuse_handler]} 전달
langfuse_handler = None
if os.environ.get("LANGFUSE_SECRET_KEY"):
    from langfuse.langchain import CallbackHandler
    langfuse_handler = CallbackHandler()
    print(f"Langfuse tracing ON \u2014 {os.environ.get('LANGFUSE_HOST', '')}")

# Langfuse config: pass to invoke/stream/batch calls
lf_config = {"callbacks": [langfuse_handler]} if langfuse_handler else {}


Langfuse tracing ON — https://lf.ddok.ai


## 5.2 단기 메모리: InMemorySaver

단기 메모리는 **하나의 대화 세션** 내에서 이전 메시지를 기억하는 메커니즘입니다.

- `InMemorySaver`는 체크포인터(checkpointer)로서 에이전트의 상태를 메모리에 저장합니다.
- `thread_id`를 사용하여 서로 다른 대화 세션을 구분합니다.
- 같은 `thread_id`를 사용하면 이전 대화 컨텍스트가 유지됩니다.

In [3]:
from langchain.agents import create_agent
from langchain.tools import tool
from langgraph.checkpoint.memory import InMemorySaver

@tool
def get_time() -> str:
    """현재 시간을 가져옵니다."""
    from datetime import datetime
    return datetime.now().strftime("%H:%M:%S")

checkpointer = InMemorySaver()
agent = create_agent(
    model=model,
    tools=[get_time],
    system_prompt="당신은 유용한 어시스턴트입니다. 대화 맥락을 기억하세요.",
    checkpointer=checkpointer,
)

config = {"configurable": {"thread_id": "session-1"}}

# 첫 번째 대화
result1 = agent.invoke(
    {"messages": [{"role": "user", "content": "제 이름은 앨리스입니다. 지금 몇 시인가요?"}]},
    config={**config, **lf_config},
)
print("응답 1:", result1["messages"][-1].content)

# 두 번째 대화 — 이전 대화를 기억합니다
result2 = agent.invoke(
    {"messages": [{"role": "user", "content": "제 이름이 뭔가요?"}]},
    config={**config, **lf_config},
)
print("응답 2:", result2["messages"][-1].content)

응답 1: 앨리스님, 지금은 00시 1분 57초입니다. 다른 궁금한 점 있으신가요?


응답 2: 고객님의 이름은 앨리스입니다. 혹시 더 궁금하신 점 있으신가요?


## 5.3 다른 thread_id로 독립된 대화

서로 다른 `thread_id`를 사용하면 완전히 **독립된 대화 세션**이 생성됩니다. 이전 세션의 컨텍스트는 공유되지 않습니다.

In [4]:
config_b = {"configurable": {"thread_id": "session-2"}}

result3 = agent.invoke(
    {"messages": [{"role": "user", "content": "제 이름을 아시나요?"}]},
    config={**config_b, **lf_config},
)
print("다른 세션 응답:", result3["messages"][-1].content)
print("→ 다른 thread_id이므로 이전 대화를 모릅니다")

다른 세션 응답: 아직 님의 이름은 알려주지 않으셨어요. 원하신다면 알려주실 수 있나요?
→ 다른 thread_id이므로 이전 대화를 모릅니다


## 5.4 메시지 트리밍

대화가 길어지면 토큰 수가 증가하여 비용과 성능에 영향을 줍니다. **메시지 트리밍**을 사용하면 토큰 예산 내에서 가장 관련성 높은 메시지만 유지할 수 있습니다.

- `trim_messages`: 최근 N개의 메시지 또는 토큰 예산 내의 메시지만 유지합니다.
- `strategy="last"`: 가장 최근 메시지를 우선 유지합니다.
- `include_system=True`: 시스템 메시지는 항상 포함합니다.

In [5]:
from langchain.messages import trim_messages

# 트리밍 함수 설정: 최근 N개 메시지만 유지
trimmer = trim_messages(
    max_tokens=1000,
    strategy="last",
    token_counter=model,
    include_system=True,
    allow_partial=False,
)

# 트리밍을 미들웨어로 적용하는 예시
from langchain.agents.middleware import before_model

@before_model
def trim_context(request):
    """토큰 예산에 맞게 메시지를 트리밍합니다."""
    trimmed = trimmer.invoke(request.state["messages"], config=lf_config)
    return request.override(messages=trimmed)

agent_trimmed = create_agent(
    model=model,
    tools=[get_time],
    system_prompt="당신은 유용한 어시스턴트입니다.",
    middleware=[trim_context],
    checkpointer=InMemorySaver(),
)

print("메시지 트리밍 에이전트 생성 완료")

메시지 트리밍 에이전트 생성 완료


## 5.5 장기 메모리: InMemoryStore

장기 메모리는 **대화 세션 간에 지속되는** 정보를 저장합니다.

- `InMemoryStore`는 키-값 저장소로서 사용자 선호도, 설정 등을 저장합니다.
- 도구의 `ToolRuntime` 파라미터를 통해 스토어에 접근할 수 있습니다.
- `thread_id`와 무관하게 모든 세션에서 동일한 데이터에 접근 가능합니다.

단기 메모리와 장기 메모리의 차이:

| 구분 | 단기 메모리 (Checkpointer) | 장기 메모리 (Store) |
|------|--------------------------|--------------------|
| 범위 | 하나의 `thread_id` 내 | 모든 세션에 걸쳐 |
| 저장 대상 | 대화 메시지 히스토리 | 사용자 선호도, 학습 데이터 |
| 수명 | 세션 종료 시 (또는 영구) | 명시적 삭제 전까지 영구 |
| 접근 방식 | 자동 (에이전트 내부) | 도구를 통해 명시적 |

In [6]:
from langgraph.store.memory import InMemoryStore
from langchain.tools import tool, ToolRuntime

store = InMemoryStore()

@tool
def save_preference(key: str, value: str, runtime: ToolRuntime) -> str:
    """사용자 선호도를 장기 메모리에 저장합니다."""
    runtime.store.put(("preferences",), key, {"value": value})
    return f"선호도 저장 완료: {key} = {value}"

@tool
def get_preference(key: str, runtime: ToolRuntime) -> str:
    """장기 메모리에서 사용자 선호도를 조회합니다."""
    result = runtime.store.get(("preferences",), key)
    if result:
        return f"선호도 {key} = {result.value.get('value', '찾을 수 없음')}"
    return f"{key}에 대한 선호도를 찾을 수 없습니다"

memory_agent = create_agent(
    model=model,
    tools=[save_preference, get_preference],
    system_prompt="도구를 사용하여 사용자 선호를 저장하고 조회할 수 있습니다.",
    store=store,
    checkpointer=InMemorySaver(),
)

config = {"configurable": {"thread_id": "memory-test"}}

# 선호도 저장
result = memory_agent.invoke(
    {"messages": [{"role": "user", "content": "제가 좋아하는 색은 파랑, 좋아하는 음식은 피자로 저장해 주세요."}]},
    config={**config, **lf_config},
)
print("저장:", result["messages"][-1].content)

# 새 세션에서 선호도 조회
config2 = {"configurable": {"thread_id": "memory-test-2"}}
result2 = memory_agent.invoke(
    {"messages": [{"role": "user", "content": "제가 좋아하는 색은 뭔가요?"}]},
    config={**config2, **lf_config},
)
print("조회:", result2["messages"][-1].content)

저장: 당신이 좋아하는 색은 '파랑', 좋아하는 음식은 '피자'로 저장해 두었습니다. 앞으로 필요할 때 알려드릴 수 있어요!


조회: 당신이 좋아하는 색은 "파랑"입니다. 

다른 궁금한 점이나 좋아하는 것이 있으면 언제든 말씀해 주세요!


## 5.6 스트리밍 모드

에이전트의 실행 과정을 **실시간으로 관찰**할 수 있는 스트리밍 기능을 제공합니다. 용도에 따라 다양한 스트리밍 모드를 선택할 수 있습니다.

### 스트리밍 모드 비교표

| 모드 | 설명 | 용도 |
|------|------|------|
| `values` | 각 단계의 전체 상태 | 디버깅, 상태 추적 |
| `updates` | 각 노드의 업데이트만 | 진행 상황 표시 |
| `messages` | 메시지 토큰 단위 | 채팅 UI |
| `custom` | 사용자 정의 이벤트 | 커스텀 진행률 |

In [7]:
# stream_mode="updates" — 각 노드의 업데이트
print("=== stream_mode='updates' ===")
for chunk in agent.stream(
    {"messages": [{"role": "user", "content": "지금 몇 시인가요?"}]},
    config={**{"configurable": {"thread_id": "stream-1"}}, **lf_config},
    stream_mode="updates",
):
    for node, update in chunk.items():
        print(f"[{node}]", end=" ")
        if "messages" in update:
            for msg in update["messages"]:
                content = msg.content if hasattr(msg, 'content') else str(msg)
                if content:
                    print(content[:200])

=== stream_mode='updates' ===


[model] [tools] 00:02:08


[model] 현재 시간은 00시 02분 08초입니다.


In [8]:
# stream_mode="messages" — 토큰 단위 스트리밍
print("=== stream_mode='messages' ===")
for msg, metadata in agent.stream(
    {"messages": [{"role": "user", "content": "짧은 농담 하나 해주세요."}]},
    config={**{"configurable": {"thread_id": "stream-2"}}, **lf_config},
    stream_mode="messages",
):
    if hasattr(msg, 'content') and msg.content:
        print(msg.content, end="", flush=True)
print()

=== stream_mode='messages' ===


왜

 바

나

나는

...

서

요

!

 🍌

😄

In [9]:
# stream_mode="values" — 각 단계의 전체 상태 스냅샷
print("=== stream_mode='values' ===")
for state_snapshot in agent.stream(
    {"messages": [{"role": "user", "content": "지금 몇 시인가요?"}]},
    config={**{"configurable": {"thread_id": "stream-3"}}, **lf_config},
    stream_mode="values",
):
    msgs = state_snapshot["messages"]
    last_msg = msgs[-1]
    role = getattr(last_msg, "type", "unknown")
    content = last_msg.content if hasattr(last_msg, "content") else str(last_msg)
    tool_calls = getattr(last_msg, "tool_calls", None)
    print(f"[전체 상태] 메시지 수: {len(msgs)} | 마지막 메시지 역할: {role}")
    if content:
        print(f"  내용: {content[:200]}")
    if tool_calls:
        print(f"  도구 호출: {[tc['name'] for tc in tool_calls]}")

=== stream_mode='values' ===
[전체 상태] 메시지 수: 1 | 마지막 메시지 역할: human
  내용: 지금 몇 시인가요?


[전체 상태] 메시지 수: 2 | 마지막 메시지 역할: ai
  도구 호출: ['get_time']
[전체 상태] 메시지 수: 3 | 마지막 메시지 역할: tool
  내용: 00:02:11


[전체 상태] 메시지 수: 4 | 마지막 메시지 역할: ai
  내용: 지금 시간은 00시 2분입니다.


### stream_mode="custom" 참고

`stream_mode="custom"`은 사용자 정의 이벤트를 스트리밍하는 모드입니다. 이 모드는 `create_agent`로 생성된 에이전트에서 직접 사용할 수 없으며, **LangGraph의 저수준 API**(`StateGraph`)에서 `StreamWriter`를 통해 커스텀 이벤트를 수동으로 발행해야 합니다.

```python
# LangGraph StateGraph 수준에서의 사용 예시 (참고용)
from langgraph.graph import StateGraph

def my_node(state, writer):  # StreamWriter가 주입됨
    writer("progress", {"step": 1, "status": "processing"})
    # ... 처리 로직 ...
    writer("progress", {"step": 2, "status": "done"})
    return state
```

`create_agent`를 사용하는 경우, 커스텀 진행률 표시가 필요하다면 `stream_mode="updates"`와 미들웨어를 조합하는 방식을 권장합니다.

## 5.7 요약

이 노트북에서 학습한 내용을 정리합니다:

| 개념 | 구현 | 설명 |
|------|------|------|
| **단기 메모리** | `InMemorySaver` + `thread_id` | 하나의 대화 세션 내 컨텍스트 유지 |
| **세션 격리** | 다른 `thread_id` 사용 | 독립된 대화 세션 관리 |
| **메시지 트리밍** | `trim_messages` + 미들웨어 | 토큰 예산 내 메시지 제한 |
| **장기 메모리** | `InMemoryStore` + `ToolRuntime` | 대화 간 지속되는 사용자 데이터 |
| **스트리밍 (values)** | `stream_mode="values"` | 각 단계의 전체 상태 스냅샷 |
| **스트리밍 (updates)** | `stream_mode="updates"` | 노드별 업데이트 확인 |
| **스트리밍 (messages)** | `stream_mode="messages"` | 토큰 단위 실시간 출력 |
| **스트리밍 (custom)** | `stream_mode="custom"` | LangGraph `StateGraph` 수준에서만 사용 가능 |

**핵심 포인트:**
- 단기 메모리는 `thread_id`로 격리되며, 같은 세션 내에서만 컨텍스트가 유지됩니다.
- 장기 메모리는 `InMemoryStore`를 통해 세션 간에 공유됩니다.
- `stream_mode="values"`는 각 단계의 전체 상태를 반환하여 디버깅에 유용합니다.
- `stream_mode="custom"`은 `create_agent`에서는 직접 사용할 수 없으며, LangGraph의 `StateGraph` API가 필요합니다.
- 스트리밍 모드를 적절히 선택하면 사용자 경험을 크게 향상시킬 수 있습니다.

### 다음 단계
→ **[06_middleware.ipynb](./06_middleware.ipynb)**: 미들웨어와 가드레일을 배웁니다.
